In [4]:
import pandas as pd
from datetime import datetime
import sqlite3
import random
from sqlalchemy import create_engine, text
import os

In [5]:
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "vendor_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
conn = create_engine(DATABASE_URL)

In [3]:
# conn = sqlite3.connect("./my_database.db")

In [6]:
vendor_ids = pd.read_sql("select distinct(vendor_id) from ticket", con=conn)

In [10]:
test_vendor = vendor_ids.iloc[2]['vendor_id']
test_vendor

'87557121-007b-4e66-9a6e-733b6e2bc363'

In [7]:
df = pd.read_sql("select * from ticket", con=conn)
df.head()

,ticket_id,work_order_id,invoice_id,description,assigned_contractor,priority,status,vendor_id,start_time,due_date,...,estimated_quantity,unit,special_requirements,anomaly_flag,anomaly_reason,created_at,updated_at,created_by,updated_by,additional_information
0,e1d22db6-ade8-410a-8bdd-7da3d0ea8471,f5d1a1bf-5cc7-43db-9b4a-3792258adb1b,NaN,Question garden movie. Memory require type sea...,NaN,MEDIUM,UNASSIGNED,87557121-007b-4e66-9a6e-733b6e2bc363,NaT,2025-02-25 11:46:23.337211+00:00,...,82.64,visits,Partner whose any standard rate woman.,True,Behind far already here top whose discuss.,2025-02-22 10:36:15.526145+00:00,2026-04-12 23:49:46.316200+00:00,246c9cfa-c9ba-4496-8c25-9c9ac44892c2,a15b43e2-d9ff-44b8-a875-5c98702ffac7,"{'source': 'sample_generator', 'has_photo': Fa..."
1,b80c7be4-850c-417a-a684-e3ec78715051,1a596082-24a1-4d8f-8126-21edb0348a45,NaN,Believe candidate office. Summer evidence memb...,35a22921-0807-4859-ba89-2ebae3c25783,HIGH,ASSIGNED,0709cc65-6c9f-4f4d-8ae3-f0bbc2a2a1a8,NaT,2025-09-05 15:50:26.923977+00:00,...,57.93,days,Sea general experience idea note public.,True,NaN,2025-08-30 04:46:36.894150+00:00,2026-04-12 23:49:46.316403+00:00,47c914c6-9d00-465e-af18-52e9267e30ee,4b87c123-ea0c-4a20-957e-f256d1c7602b,"{'source': 'sample_generator', 'has_photo': True}"
2,d5a0fd7f-42d7-47e3-b5ac-d1a3f5882a3b,2c2aadbf-1ba2-46c4-aadd-fd3d7d1942de,NaN,Your oil can computer. About region order shou...,0708c408-e916-4c6f-8f20-32780df16bcb,LOW,ASSIGNED,600ad80b-c837-4db6-b132-5c5f8b995ef6,NaT,2025-12-28 08:52:30.617977+00:00,...,55.68,visits,Change success blood again real only.,False,NaN,2025-12-24 13:25:03.770068+00:00,2026-04-12 23:49:46.316491+00:00,4ea9ea0c-6673-4643-88d7-fa31c5686d64,bd081e16-bc11-4549-8627-6d46e96d3ed7,"{'source': 'sample_generator', 'has_photo': True}"
3,6e0962d6-4212-4a02-aba7-6212b8859232,4f2dd236-ce26-4769-8ad4-1b340846fd72,NaN,While himself scientist hot analysis. Choice i...,c8169bf2-0bae-4971-b563-b7197d0ae052,HIGH,ASSIGNED,600ad80b-c837-4db6-b132-5c5f8b995ef6,NaT,2025-10-21 21:20:27.381212+00:00,...,70.39,hours,Response management work college.,False,Seem really military field road.,2025-10-17 03:25:33.605479+00:00,2026-04-12 23:49:46.316580+00:00,22f2e747-2de3-45cc-a9b8-9248c0d8b07f,f5f27d96-725b-4f88-9ea5-f98db439b7af,"{'source': 'sample_generator', 'has_photo': True}"
4,7f0efafc-b797-49ce-afde-1ef439453ed7,9dfc6b0e-94b6-42aa-b76d-c868a8760cb2,NaN,Late hand way network good some respond or. Co...,b560b26d-e5d4-489e-bcf3-88877d42cb92,LOW,COMPLETED,87557121-007b-4e66-9a6e-733b6e2bc363,2025-07-07 15:09:52.882270+00:00,2025-07-12 03:43:38.662657+00:00,...,82.49,hours,Size all eight large create left room.,False,NaN,2025-07-05 17:02:14.505882+00:00,2026-04-12 23:49:46.316706+00:00,8b8807f4-c988-4fd1-8838-a2d500359eb1,f078e66c-2df3-4350-940b-8e9d7dbf7bc7,"{'source': 'sample_generator', 'has_photo': True}"


In [8]:
def unassigned_work_orders(vendor_id):
    # df = pd.read_sql(
    #     f"""
    #     select count(*) as unassigned_work_orders
    #     from work_orders
    #     where assigned_vendor='{vendor_id}'"""
    # , con=conn)
    df = pd.read_sql(
        f"""
        select work_order_id, count(*) as "unassigned tickets"
        from ticket t
        join work_orders wo
        using (work_order_id)
        where t.status = 'UNASSIGNED' and wo.assigned_vendor = '{vendor_id}'
        group by 1;
        """, con=conn
    )
    return {'vendor_id': vendor_id, 'unassigned tickets': df['unassigned tickets'].sum()}

In [11]:
unassigned_work_orders(test_vendor)

{'vendor_id': '87557121-007b-4e66-9a6e-733b6e2bc363',
 'unassigned tickets': np.int64(94)}

In [12]:
def pending_tickets(vendor_id):
    df = pd.read_sql(f"""
    select count(*) as active_tickets
    from ticket 
    where vendor_id='{vendor_id}' and (status = 'IN_PROGRESS' or status = 'ASSIGNED')
    """, con=conn)
    count = df['active_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'active tickets': count}

In [13]:
pending_tickets(test_vendor)

{'vendor_id': '87557121-007b-4e66-9a6e-733b6e2bc363',
 'active tickets': np.int64(217)}

In [30]:
df = pd.read_sql(
    f"""select * 
    from ticket 
    where vendor_id='{test_vendor}' and completed_at is not null and created_at > NOW() - interval '12 months'"""
    , con=conn, parse_dates=['assigned_at', 'completed_at'])

df.head(1)

,ticket_id,work_order_id,invoice_id,description,assigned_contractor,priority,status,vendor_id,start_time,due_date,...,estimated_quantity,unit,special_requirements,anomaly_flag,anomaly_reason,created_at,updated_at,created_by,updated_by,additional_information
0,7f0efafc-b797-49ce-afde-1ef439453ed7,9dfc6b0e-94b6-42aa-b76d-c868a8760cb2,NaN,Late hand way network good some respond or. Co...,b560b26d-e5d4-489e-bcf3-88877d42cb92,LOW,COMPLETED,87557121-007b-4e66-9a6e-733b6e2bc363,2025-07-07 15:09:52.882270+00:00,2025-07-12 03:43:38.662657+00:00,...,82.49,hours,Size all eight large create left room.,False,NaN,2025-07-05 17:02:14.505882+00:00,2026-04-12 23:49:46.316706+00:00,8b8807f4-c988-4fd1-8838-a2d500359eb1,f078e66c-2df3-4350-940b-8e9d7dbf7bc7,"{'source': 'sample_generator', 'has_photo': True}"


In [31]:
df['completion_week'] = df['completed_at'].dt.to_period('W').apply(lambda r: r.end_time).dt.date

C:\Users\danie\AppData\Local\Temp\ipykernel_14896\1373384851.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['completion_week'] = df['completed_at'].dt.to_period('W').apply(lambda r: r.end_time).dt.date


In [32]:
complete_by_week = df.groupby('completion_week')['ticket_id'].count().reset_index().sort_values(by='ticket_id', ascending=False)
complete_by_week.rename(columns={'ticket_id': 'ticket_count'}, inplace=True)
complete_by_week

,completion_week,ticket_count
2,2025-07-13,16
3,2025-07-20,9
7,2026-02-01,6
4,2025-07-27,3
1,2025-05-11,2
0,2025-05-04,1
5,2025-09-07,1
6,2026-01-25,1
